In [56]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import json
import os
import sys

from preprocessing.ecg_preprocessor import ECGPreprocessor
from preprocessing.lead_processor import LeadProcessor

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [57]:
test_ecg_path = os.path.expanduser("~/UNI/CORSI/Visione_Artificiale/progetto/datasets/PTB/s0001_re.csv")
lead_names = ["i","ii","iii","avl","avr","avf","v1","v2","v3","v4","v5","v6"]
df = pd.read_csv(test_ecg_path)[lead_names]
data_in = df.to_numpy().T
freq_hz = 500

lead_to_test = 0

preprocessor = ECGPreprocessor(freq_hz)
results_post_filters, m_detrend, error_preprocessing = preprocessor.run(data_in)

lead_i = LeadProcessor(data_in[lead_to_test], lead_idx=lead_to_test, freq_hz=freq_hz)
lead_i.preprocess_single_lead()
pre_I = lead_i.to_dict()
print(pre_I)


{'loessRPeaksEnd': array([  303.,  1128.,  1923.,  2728.,  3540.,  4398.,  5247.,  6064.,
        6922.,  7778.,  8602.,  9440., 10275., 11112., 11927., 12756.,
       13582., 14404., 15215., 16041., 16865., 17691., 18504., 19336.,
       20170., 21008., 21836., 22680., 23501., 24321., 25142., 25978.,
       26796., 27605., 28422., 29260., 30093., 30903., 31714., 32559.,
       33396., 34208., 35037., 35866., 36689., 37504., 38334.]), 'loessSegEnd': array([[    0,   798],
       [  799,  1605],
       [ 1606,  2406],
       [ 2407,  3216],
       [ 3217,  4055],
       [ 4056,  4908],
       [ 4909,  5738],
       [ 5739,  6579],
       [ 6580,  7436],
       [ 7437,  8273],
       [ 8274,  9105],
       [ 9106,  9941],
       [ 9942, 10778],
       [10779, 11601],
       [11602, 12425],
       [12426, 13252],
       [13253, 14076],
       [14077, 14891],
       [14892, 15711],
       [15712, 16536],
       [16537, 17361],
       [17362, 18179],
       [18180, 19004],
       [19005, 19

In [58]:
# Load R json outputs
with open("test_R_porting/r_test_pre_I.json", "r") as f: r_pre_I = json.load(f)
with open("test_R_porting/r_test_pos_results.json", "r") as f: r_pos_results = json.load(f)
with open("test_R_porting/r_test_m_detrend.json", "r") as f: r_m_detrend = json.load(f)

In [59]:
# TEST LOESSDATA (segnale ECG con rimozione baseline wander)

r_loessData = np.array(r_pre_I["loessData"])
python_loessData = pre_I["loessData"]

try:
    np.testing.assert_allclose(python_loessData, r_loessData, rtol=1e-5, atol=1e-5)
    print("✅ loessData matches!")
except AssertionError as e:
    print("❌ loessData DOES NOT MATCH!")
    print(e)

✅ loessData matches!


In [60]:
# TEST PICCHI R

r_loessRPeaksEnd = np.array(r_pre_I["loessRPeaksEnd"])
python_loessRPeaksEnd = pre_I["loessRPeaksEnd"]

try:
    np.testing.assert_allclose(python_loessRPeaksEnd, r_loessRPeaksEnd - 1, rtol=1e-5, atol=1e-5)
    print("✅ loessRPeaksEnd matches!")
except AssertionError as e:
    print("❌ loessRPeaksEnd DOES NOT MATCH!")
    print(e)

✅ loessRPeaksEnd matches!


In [61]:
# TEST SEGMENTI QRS

r_loessSegEnd = np.array(r_pre_I["loessSegEnd"])
python_loessSegEnd = pre_I["loessSegEnd"]

try:
    np.testing.assert_allclose(python_loessSegEnd, r_loessSegEnd - 1, rtol=1e-5, atol=1e-5)
    print("✅ loessSegEnd matches!")
except AssertionError as e:
    print("❌ loessSegEnd DOES NOT MATCH!")
    print(e)

✅ loessSegEnd matches!


In [62]:
# TEST m_detrend (segnale pulito)

r_m_detrend_np = np.array(r_m_detrend).T
try:
    np.testing.assert_allclose(m_detrend, r_m_detrend_np, rtol=1e-5, atol=1e-5)
    print("✅ m_detrend matches!")
except AssertionError as e:
    print("❌ m_detrend DOES NOT MATCH!")
    print(e)

✅ m_detrend matches!


In [63]:
final_output_R = []

for beat in r_pos_results:
    # | QRS location | beat start | beat end |
    final_output_R.append(beat[1:4])

der = 0
beats_der = results_post_filters[der]['loessRPeaksEnd']
start_end_der = results_post_filters[der]['loessSegEnd']

final_output_python = []

for i in range(len(beats_der)):
    final_output_python.append([int(beats_der[i])+1, int(start_end_der[i][0])+1, int(start_end_der[i][1])+1])

In [64]:
try:
    np.testing.assert_allclose(final_output_python, final_output_R, rtol=1e-5, atol=1e-5)
    print("✅ pos_results matches!")
except AssertionError as e:
    print("❌ pos_results DOES NOT MATCH!")

print(final_output_R)
print(final_output_python)


❌ pos_results DOES NOT MATCH!
[[303, 1, 798], [1128, 799, 1605], [1923, 1606, 2406], [2728, 2407, 3216], [3541, 3217, 4056], [4398, 4057, 4908], [5247, 4909, 5738], [6065, 5739, 6580], [6922, 6581, 7436], [7778, 7437, 8273], [8603, 8274, 9106], [9441, 9107, 9941], [10273, 9942, 10777], [11112, 10778, 11601], [11926, 11602, 12425], [12757, 12426, 13252], [13582, 13253, 14076], [14405, 14077, 14892], [15216, 14893, 15712], [16042, 15713, 16537], [16866, 16538, 17362], [17692, 17363, 18180], [18505, 18181, 19004], [19336, 19005, 19837], [20170, 19838, 20673], [21008, 20674, 21506], [21837, 21507, 22343], [22680, 22344, 23174], [23502, 23175, 23994], [24321, 23995, 24815], [25143, 24816, 25644], [25978, 25645, 26469], [26796, 26470, 27282], [27606, 27283, 28097], [28423, 28098, 28926], [29261, 28927, 29761], [30094, 29762, 30580], [30904, 30581, 31391], [31715, 31392, 32222], [32559, 32223, 33062], [33397, 33063, 33883], [34207, 33884, 34706], [35038, 34707, 35535], [35866, 35536, 36361], 